# Black-Scholes Pricing Baseline

Black-Scholes is a baseline model for valuing European call and put options. The model links an option's theoretical value to the current stock price, strike price, time to expiry, risk-free rate, and volatility.

The pricing functions in this notebook are used to study how option values respond to changes in volatility and time to expiry.

## Model inputs

| Symbol | Meaning |
|---|---|
| \(S\) | Current stock price |
| \(K\) | Strike price |
| \(T\) | Time to expiry, measured in years |
| \(r\) | Continuously compounded risk-free rate |
| \(\sigma\) | Annualized volatility |

The model assumes European exercise, constant volatility, frictionless markets, no transaction costs, and continuous hedging. These assumptions make the model tractable, but real option markets often violate them.

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

## Standard normal functions

The Black-Scholes formula uses the standard normal cumulative distribution function \(N(x)\). The implementation below uses the error function from Python's standard library.

In [ ]:
def normal_cdf(x: float) -> float:
    """Return the cumulative probability for a standard normal variable."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def normal_pdf(x: float) -> float:
    """Return the density of a standard normal variable."""
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

## Pricing formula

For a European call:

\[
C = S N(d_1) - K e^{-rT} N(d_2)
\]

For a European put:

\[
P = K e^{-rT} N(-d_2) - S N(-d_1)
\]

where

\[
d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma \sqrt{T}}
\]

\[
d_2 = d_1 - \sigma \sqrt{T}
\]

The term \(K e^{-rT}\) discounts the strike payment back to present value. The \(N(d_1)\) and \(N(d_2)\) terms come from the standard normal distribution used in the model.

In [ ]:
def black_scholes_price(
    S: float,
    K: float,
    T: float,
    r: float,
    sigma: float,
    option_type: str = "call",
) -> float:
    """Price a European call or put using the Black-Scholes formula."""
    if S <= 0:
        raise ValueError("S must be positive.")
    if K <= 0:
        raise ValueError("K must be positive.")
    if sigma < 0:
        raise ValueError("sigma cannot be negative.")

    option_type = option_type.lower()
    if option_type not in {"call", "put"}:
        raise ValueError("option_type must be 'call' or 'put'.")

    if T <= 0:
        if option_type == "call":
            return max(S - K, 0.0)
        return max(K - S, 0.0)

    if sigma == 0:
        discounted_strike = K * math.exp(-r * T)
        if option_type == "call":
            return max(S - discounted_strike, 0.0)
        return max(discounted_strike - S, 0.0)

    sqrt_T = math.sqrt(T)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    if option_type == "call":
        return S * normal_cdf(d1) - K * math.exp(-r * T) * normal_cdf(d2)

    return K * math.exp(-r * T) * normal_cdf(-d2) - S * normal_cdf(-d1)

## Baseline valuation

The baseline case uses an at-the-money option with six months to expiry.

In [ ]:
S = 100.0
K = 100.0
T = 0.5
r = 0.04
sigma = 0.25

baseline_prices = pd.DataFrame(
    {
        "option_type": ["call", "put"],
        "price": [
            black_scholes_price(S, K, T, r, sigma, "call"),
            black_scholes_price(S, K, T, r, sigma, "put"),
        ],
    }
)

baseline_prices["price"] = baseline_prices["price"].round(4)
baseline_prices

## Sensitivity to volatility

Higher volatility increases option value because the distribution of possible future stock prices becomes wider. For both calls and puts, a wider distribution increases the chance of a large payoff while the downside for the option holder remains limited to the premium paid.

In [ ]:
volatility_values = [0.10, 0.20, 0.30, 0.40, 0.60]

volatility_sensitivity = pd.DataFrame(
    {
        "volatility": volatility_values,
        "call_price": [
            black_scholes_price(S, K, T, r, vol, "call")
            for vol in volatility_values
        ],
        "put_price": [
            black_scholes_price(S, K, T, r, vol, "put")
            for vol in volatility_values
        ],
    }
)

volatility_sensitivity["volatility"] = volatility_sensitivity["volatility"].map(lambda x: f"{x:.0%}")
volatility_sensitivity[["call_price", "put_price"]] = volatility_sensitivity[
    ["call_price", "put_price"]
].round(4)

volatility_sensitivity

In [ ]:
volatility_plot = pd.DataFrame(
    {
        "volatility": volatility_values,
        "call_price": [
            black_scholes_price(S, K, T, r, vol, "call")
            for vol in volatility_values
        ],
        "put_price": [
            black_scholes_price(S, K, T, r, vol, "put")
            for vol in volatility_values
        ],
    }
)

plt.figure(figsize=(8, 5))
plt.plot(volatility_plot["volatility"], volatility_plot["call_price"], marker="o", label="Call")
plt.plot(volatility_plot["volatility"], volatility_plot["put_price"], marker="o", label="Put")
plt.xlabel("Annualized Volatility")
plt.ylabel("Option Price")
plt.title("Option Price Sensitivity to Volatility")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Sensitivity to time to expiry

More time to expiry generally increases option value because the underlying stock has more time to move. This effect is strongest when the option has meaningful probability of finishing in the money.

In [ ]:
time_values = [
    ("7 days", 7 / 365),
    ("30 days", 30 / 365),
    ("90 days", 90 / 365),
    ("180 days", 180 / 365),
    ("1 year", 1.0),
    ("2 years", 2.0),
]

time_sensitivity = pd.DataFrame(
    {
        "time_to_expiry": [label for label, _ in time_values],
        "years": [years for _, years in time_values],
        "call_price": [
            black_scholes_price(S, K, years, r, sigma, "call")
            for _, years in time_values
        ],
        "put_price": [
            black_scholes_price(S, K, years, r, sigma, "put")
            for _, years in time_values
        ],
    }
)

time_sensitivity[["years", "call_price", "put_price"]] = time_sensitivity[
    ["years", "call_price", "put_price"]
].round(4)

time_sensitivity

In [ ]:
time_plot = pd.DataFrame(
    {
        "years": [years for _, years in time_values],
        "call_price": [
            black_scholes_price(S, K, years, r, sigma, "call")
            for _, years in time_values
        ],
        "put_price": [
            black_scholes_price(S, K, years, r, sigma, "put")
            for _, years in time_values
        ],
    }
)

plt.figure(figsize=(8, 5))
plt.plot(time_plot["years"], time_plot["call_price"], marker="o", label="Call")
plt.plot(time_plot["years"], time_plot["put_price"], marker="o", label="Put")
plt.xlabel("Time to Expiry in Years")
plt.ylabel("Option Price")
plt.title("Option Price Sensitivity to Time to Expiry")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Model limitations

Black-Scholes is useful as a benchmark, but the assumptions are simplified.

The model uses one constant volatility value, while real option chains often show different implied volatilities across strikes and expiries. The model also assumes frictionless trading, while real markets have bid-ask spreads, transaction costs, limited liquidity, and stale quotes. Continuous hedging is another simplifying assumption, since real hedges can only be rebalanced at discrete times.

These limitations are important because they affect implied volatility estimates, Greek sensitivity, and hedging performance when the model is applied to market data.